#15.777 Homework 2 (Spring 2024): Natural Langugage Processing

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras

keras.utils.set_random_seed(42)

# Introduction

This homework assignment will ask you to build models using BoW, GloVE and BERT.

We will work with a famous dataset in natural langauge processing called **20 Newsgroup**,  which consists of posts from an online forum under certain topics such as politics, religion, sports...etc. As the name suggests, there are a total of 20 topics in this dataet. The 20 Newsgroup dataset is a popular benchmark for text classification algorithms.

The entire dataset is quite large. To ensure training takes a reasonable amount of time, we will only choose 6 out of the 20 topics, including topics like religion, space (astronomy) and medicine.

In [ ]:
from sklearn.datasets import fetch_20newsgroups

newsgroups_train = fetch_20newsgroups(subset='train', categories = ['alt.atheism', 'talk.religion.misc', 'comp.graphics', 'sci.space', 'sci.med', 'rec.autos'])
newsgroups_test = fetch_20newsgroups(subset='test', categories = ['alt.atheism', 'talk.religion.misc', 'comp.graphics', 'sci.space', 'sci.med', 'rec.autos'])

train_df = pd.DataFrame({'text': newsgroups_train.data, 'label': newsgroups_train.target})
test_df = pd.DataFrame({'text': newsgroups_test.data, 'label': newsgroups_test.target})

print(f"""
Train samples: {train_df.shape[0]}
Test samples: {test_df.shape[0]}
""")

train_df.head(10)


Train samples: 3222
Test samples: 2145



,text,label
0,From: boylan@pi.eai.iastate.edu (Terran Boylan...,1
1,From: I3150101@dbstu1.rz.tu-bs.de (Benedikt Ro...,0
2,From: snichols@adobe.com (Sherri Nichols)\nSub...,3
3,From: johnm@spudge.lonestar.org (John Munsch)\...,1
4,From: Nanci Ann Miller <nm0w+@andrew.cmu.edu>\...,0
5,From: nsmca@aurora.alaska.edu\nSubject: 30826\...,4
6,From: geb@cs.pitt.edu (Gordon Banks)\nSubject:...,3
7,From: higgins@fnalf.fnal.gov (Bill Higgins-- B...,4
8,From: mmm@cup.portal.com (Mark Robert Thorson)...,3
9,From: daniel@lclark.edu (Daniel Snodgrass)\nSu...,1


The distribution of labels across these 6 classes is fairly balanced.



In [ ]:
train_df['label'].value_counts() / train_df.shape[0]

3    0.184358
2    0.184358
4    0.184047
1    0.181254
0    0.148976
5    0.117008
Name: label, dtype: float64

Let's convert our dependent variable into a 1-hot-encoded vector.

In [ ]:
# Let's turn the target into a dummy vector
y_train = pd.get_dummies(train_df['label']).to_numpy()
y_test = pd.get_dummies(test_df['label']).to_numpy()

y_train[:10]

array([[0, 1, 0, 0, 0, 0],
       [1, 0, 0, 0, 0, 0],
       [0, 0, 0, 1, 0, 0],
       [0, 1, 0, 0, 0, 0],
       [1, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 1, 0],
       [0, 0, 0, 1, 0, 0],
       [0, 0, 0, 0, 1, 0],
       [0, 0, 0, 1, 0, 0],
       [0, 1, 0, 0, 0, 0]], dtype=uint8)

# Problem 1: Bag-of-Words (BoW) Model [30 Points]

In this problem, we will build a bag-of-words model using the text vectorization capabilities of Keras. We will then change some of the parameters of this vectorization process and see how it changes the performance.



## Part (a): Build a Base Model [15 Points]

**Text Vectorization**

Please fill in the code in the following cell. We would like to create a text vectorization layer which uses:

* Maximum of 2000 tokens.
* Unigrams
* Outputs a multi-hot encoded BoW encoding
* Converts text to lower case and strips punctuation

In [ ]:
# Set the maximum number of tokens (which is the size of the vocabulary) to 1000
max_tokens = 2000

# Configure the text vectorization layer
text_vectorization = keras.layers.TextVectorization(
    max_tokens=max_tokens,
    output_mode = 'multi_hot',
    standardize = 'lower_and_strip_punctuation',
    ngrams=1,
)

# Let's adapt the Text Vectorization layer using the training corpus
text_vectorization.adapt(train_df['text'])

# We vectorize our input with the adapted Text Vectorization layer
X_train = text_vectorization(train_df['text'])
X_test = text_vectorization(test_df['text'])

pd.DataFrame(X_train, columns = text_vectorization.get_vocabulary())

,[UNK],the,of,to,a,and,in,is,i,that,...,loving,learning,jaegerbuphybuedu,hour,helps,glutamate,finding,delta,careful,bother
0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0,1.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,1.0,1.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3217,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3218,1.0,1.0,1.0,1.0,0.0,1.0,1.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3219,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3220,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In the following cell, please build a simple Neural Network with a single hidden layer of 128 neurons and a ReLu activation function. Be sure to specify the shape of the input correctly and use the appropriate activation function for the output. Your model should have 256902 parameters. The code for compiling, training and computing the test accuracy has been written for you already.

In [ ]:
# Build a baseline NN model with one hidden layer that has 8 neurons.
inputs = keras.Input(shape=(max_tokens, )) # Size of the input layer is the size of the vocabulary
x = keras.layers.Dense(128, activation="relu")(inputs) # only hidden layer
outputs = keras.layers.Dense(y_train.shape[1], activation='softmax')(x)

bow_model = keras.Model(inputs, outputs)
bow_model.summary()

# Compile model using Adam
bow_model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy']
)

Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 2000)]            0         
                                                                 
 dense (Dense)               (None, 128)               256128    
                                                                 
 dense_1 (Dense)             (None, 6)                 774       
                                                                 
Total params: 256902 (1003.52 KB)
Trainable params: 256902 (1003.52 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


A theme that we will be investigating in this homework is the **impact of the number of training examples on the various models**. As a result, in addition to training the model on all 3000+ training examples, we will also test how our model performs with only 200 examples.

The code below shows a fitting process in which we first fit model `bow_model` to the first 200 examples from the training set `(X_train[:200], y_train[:200]`). We print the accuracy of the model on the test set. Then, we run the training procedure for another 20 epochs, this time with the full data set `(X_train, y_train`).

In [ ]:
# Fit model on the training data with 10 epochs and batch size of 32
bow_model.fit(
    x=X_train[:200], y=y_train[:200],
    epochs=20, batch_size=32,
    verbose=1,
)

print("\n*** Test accuracy with 200 Examples: %.4f ***\n" % bow_model.evaluate(x=X_test, y=y_test)[1])

# Fit model on the training data with 10 epochs and batch size of 32
bow_model.fit(
    x=X_train, y=y_train,
    epochs=20, batch_size=32,
    verbose=1,
)

print("\n*** Test accuracy with All Examples % .4f ***\n" % bow_model.evaluate(x=X_test, y=y_test)[1])

Epoch 1/20
7/7 [==============================] - 2s 11ms/step - loss: 1.7121 - accuracy: 0.3350
Epoch 2/20
7/7 [==============================] - 0s 4ms/step - loss: 1.1343 - accuracy: 0.9000
Epoch 3/20
7/7 [==============================] - 0s 3ms/step - loss: 0.7558 - accuracy: 0.9800
Epoch 4/20
7/7 [==============================] - 0s 3ms/step - loss: 0.4887 - accuracy: 0.9950
Epoch 5/20
7/7 [==============================] - 0s 3ms/step - loss: 0.3092 - accuracy: 0.9950
Epoch 6/20
7/7 [==============================] - 0s 4ms/step - loss: 0.1965 - accuracy: 1.0000
Epoch 7/20
7/7 [==============================] - 0s 3ms/step - loss: 0.1284 - accuracy: 1.0000
Epoch 8/20
7/7 [==============================] - 0s 3ms/step - loss: 0.0874 - accuracy: 1.0000
Epoch 9/20
7/7 [==============================] - 0s 4ms/step - loss: 0.0607 - accuracy: 1.0000
Epoch 10/20
7/7 [==============================] - 0s 3ms/step - loss: 0.0446 - accuracy: 1.0000
Epoch 11/20
7/7 [=====================

Please write down the test accuracy shown above and comment on the model's performance. What is the baseline performance by predicting the most frequent class?

* BoW with 200 Examples: 62.14%
* BoW with All Examples: 82.66%

**<font color='red'>Your Answer Here</font>**

_Please fill in the numbers above and replace this text with your brief comment_

_Solution. The model achieves a reasonable accuracy of 82.66%. The baseline in this problem is to predict the majority class, which acheives a 17% accuracy. Unsurprisingly, using fewer training examples leads to worse performance._


## Part (b): Exploring Hyperparameters [15 Points]

Now, let us try changing some of the hyperparameters that in the text vectorization process.

For each of the following modifications, please change the code for Problem 1(a), train the model and report the out-of-sample accuracy below. You do not need to duplicate the code from 1(a) here.

* Use bigrams instead of unigrams.
* Increase the maximum number of tokens from 2000 to 5000
* Use count vectorization instead of multi-hot.

Note that you should make each change independent of the other two changes. <font color='red'>Don't forget to undo each change before making the next </font>

Copy and paste the code from Problem 1(a) into a single cell. Then, make each of the 3 changes above and fill in the table of accuracies.




In [ ]:
# Set the maximum number of tokens (which is the size of the vocabulary) to 1000
max_tokens = 2000

# Configure the text vectorization layer
text_vectorization = keras.layers.TextVectorization(
    max_tokens=max_tokens,
    output_mode = 'multi_hot',
    standardize = 'lower_and_strip_punctuation',
    ngrams=2,
)

# Let's adapt the Text Vectorization layer using the training corpus
text_vectorization.adapt(train_df['text'])

# We vectorize our input with the adapted Text Vectorization layer
X_train = text_vectorization(train_df['text'])
X_test = text_vectorization(test_df['text'])

# Build a baseline NN model with one hidden layer that has 8 neurons.
inputs = keras.Input(shape=(max_tokens, )) # Size of the input layer is the size of the vocabulary
x = keras.layers.Dense(128, activation="relu")(inputs) # only hidden layer
outputs = keras.layers.Dense(y_train.shape[1], activation='softmax')(x)

bow_model = keras.Model(inputs, outputs)
bow_model.summary()

# Compile model using Adam
bow_model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy']
)

# Fit model on the training data with 10 epochs and batch size of 32
bow_model.fit(
    x=X_train[:200], y=y_train[:200],
    epochs=20, batch_size=32,
    verbose=1,
)

print("\n*** Test accuracy with 200 Examples: %.4f ***\n" % bow_model.evaluate(x=X_test, y=y_test)[1])

# Fit model on the training data with 10 epochs and batch size of 32
bow_model.fit(
    x=X_train, y=y_train,
    epochs=20, batch_size=32,
    verbose=1,
)

print("\n*** Test accuracy with All Examples % .4f ***\n" % bow_model.evaluate(x=X_test, y=y_test)[1])

Model: "model_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_2 (InputLayer)        [(None, 2000)]            0         
                                                                 
 dense_2 (Dense)             (None, 128)               256128    
                                                                 
 dense_3 (Dense)             (None, 6)                 774       
                                                                 
Total params: 256902 (1003.52 KB)
Trainable params: 256902 (1003.52 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________
Epoch 1/20
7/7 [==============================] - 1s 4ms/step - loss: 1.7806 - accuracy: 0.2300
Epoch 2/20
7/7 [==============================] - 0s 3ms/step - loss: 1.1125 - accuracy: 0.8450
Epoch 3/20
7/7 [==============================] - 0s 3ms/step - loss: 0.7096 - accuracy: 



Replace the following numbers with the out-of-sample accuracy you obtained from re-running Problem 1(a)'s code with the relevant modifications.

* Bigrams instead of unigrams
  * 200 Training Examples: 56%
  * All Training Examples: 78%
* Increasing max_tokens to 5000.
  * 200 Training Examples: 65%
  * All Training Examples: 87%
* Count instead of multi-hot
  * 200 Training Examples: 61%
  * All Training Examples: 83%

For each of the 3 modifications, discuss why one might expect such a change to be beneficial. Using the numbers above, comment on whether it improved, hurt or didn't affect the model's performance.

**<font color='red'>Your Answer Here</font>**

_Bigrams did not help. We may expect that pairs of words would provide a richer signal than a single word, but that seems to not be the case in this dataset. Note that we added pairs of words, but did not increase the max tokens. By adding pairs of words, we may have added some meaningless bigrams (e.g. "of the") we potentially pushed some useful unigrams out of the vocabulary space._

_Increasing the max token helped. This supports the idea that our vocabulary space of just the top 2000 tokens was not large enough._

_Using count instead of multi-hot did not help. Such a modification could help if the frequency of a word affects its class. For example, in sentiment analysis, there may be both positive and negative words present, but whichever type of word is present more often dictates the sentiment of the entire sentence._

# Problem 2: GloVe without Fine-Tuning [20 Points]

Now, we will train a model using GloVe. We will try using GloVe vectors out of box and with fine-tuning. Let's first download the GloVe vectors from the internet. The code below will take a couple minutes to run.

In [ ]:
!wget -O glove.6B.300d.zip https://www.dropbox.com/scl/fi/ree568j1h690ktpcn729b/glove.6B.300d.zip?rlkey=eq9r67js0n0vm2znvdev9rerf&dl=1
!unzip glove.6B.300d.zip

--2024-02-24 16:00:34--  https://www.dropbox.com/scl/fi/ree568j1h690ktpcn729b/glove.6B.300d.zip?rlkey=eq9r67js0n0vm2znvdev9rerf
Resolving www.dropbox.com (www.dropbox.com)... 162.125.1.18, 2620:100:6016:18::a27d:112
Connecting to www.dropbox.com (www.dropbox.com)|162.125.1.18|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://ucc382e919c04501702b703f7142.dl.dropboxusercontent.com/cd/0/inline/CN4PMd67U9OVSzis14wkErz3G9RJfgYxcxcU_hAwyhRFu9VL3a3Ee1zN56cYbVxPSDJvNQikpXHFY2tuzNKcYAGr49yWf93QOD5iegFqSwZI7g7N1aKMtT7EFbDPHk0NFy2TrR0g6p8oB1IMi-p5-uWh/file# [following]
--2024-02-24 16:00:35--  https://ucc382e919c04501702b703f7142.dl.dropboxusercontent.com/cd/0/inline/CN4PMd67U9OVSzis14wkErz3G9RJfgYxcxcU_hAwyhRFu9VL3a3Ee1zN56cYbVxPSDJvNQikpXHFY2tuzNKcYAGr49yWf93QOD5iegFqSwZI7g7N1aKMtT7EFbDPHk0NFy2TrR0g6p8oB1IMi-p5-uWh/file
Resolving ucc382e919c04501702b703f7142.dl.dropboxusercontent.com (ucc382e919c04501702b703f7142.dl.dropboxusercontent.com)... 162.125.1.15, 2

## Part (a): Load Pre-trained Vectors [10 Points]

Let's take the `glove.6B.300d.txt` file, which consists of 300-dimensional vectors for 400,000 words in English.

In [ ]:
embedding_dim = 300
path_to_glove_file = f"glove.6B.{embedding_dim}d.txt"

embeddings_index = {}
with open(path_to_glove_file) as f:
    for line in f:
        word, coefs = line.split(maxsplit=1)
        coefs = np.fromstring(coefs, "f", sep=" ")
        embeddings_index[word] = coefs

print(f"Found {len(embeddings_index)} word vectors.")

Found 400000 word vectors.


For any of these 400,000 words, we can query its 300-dimensional vector

In [ ]:
embeddings_index['hello']

array([-3.3712e-01, -2.1691e-01, -6.6365e-03, -4.1625e-01, -1.2555e+00,
       -2.8466e-02, -7.2195e-01, -5.2887e-01,  7.2085e-03,  3.1997e-01,
        2.9425e-02, -1.3236e-02,  4.3511e-01,  2.5716e-01,  3.8995e-01,
       -1.1968e-01,  1.5035e-01,  4.4762e-01,  2.8407e-01,  4.9339e-01,
        6.2826e-01,  2.2888e-01, -4.0385e-01,  2.7364e-02,  7.3679e-03,
        1.3995e-01,  2.3346e-01,  6.8122e-02,  4.8422e-01, -1.9578e-02,
       -5.4751e-01, -5.4983e-01, -3.4091e-02,  8.0017e-03, -4.3065e-01,
       -1.8969e-02, -8.5670e-02, -8.1123e-01, -2.1080e-01,  3.7784e-01,
       -3.5046e-01,  1.3684e-01, -5.5661e-01,  1.6835e-01, -2.2952e-01,
       -1.6184e-01,  6.7345e-01, -4.6597e-01, -3.1834e-02, -2.6037e-01,
       -1.7797e-01,  1.9436e-02,  1.0727e-01,  6.6534e-01, -3.4836e-01,
        4.7833e-02,  1.6440e-01,  1.4088e-01,  1.9204e-01, -3.5009e-01,
        2.6236e-01,  1.7626e-01, -3.1367e-01,  1.1709e-01,  2.0378e-01,
        6.1775e-01,  4.9075e-01, -7.5210e-02, -1.1815e-01,  1.86

Let's create our vocabulary space from the training data. We do this using the `TextVectorization` layer just like before. However, this time, we choose `output_mode=int`.

In [ ]:
max_length = 400 # For each piece of text, consider only the first 400 tokens (then truncate or pad)
max_tokens = 2000 # Our vocabulary space of tokens can be at most size 5000

text_vectorization_glove = keras.layers.TextVectorization(
    max_tokens=max_tokens,
    output_mode="int",
    output_sequence_length=max_length,
)

text_vectorization_glove.adapt(train_df['text'])

X_train_glove = text_vectorization_glove(train_df['text'])
X_test_glove = text_vectorization_glove(test_df['text'])

X_train_glove

<tf.Tensor: shape=(3222, 400), dtype=int64, numpy=
array([[  14,    1,    1, ...,    0,    0,    0],
       [  14,    1, 1334, ...,  726,    1,    1],
       [  14,    1,    1, ...,    0,    0,    0],
       ...,
       [  14,    1,   27, ...,  116,    1,    6],
       [  14,  546,  325, ...,    0,    0,    0],
       [  14,    1,    1, ...,    0,    0,    0]])>

In the output above for `X_train_glove`, each row is a training example and the columns represent the _index_ of the token. For example, the first training example starts with token 14, followed by token 1, then token 1,... and ending with the padding tokens 0.

**Question**: What English word does token 14 correspond to? Hint: A cell from the Introduction will help you here.

<font color='red'>**Your Answer Here**</font>

_Replace this with your answer and explain how you know._
_The token 14 corresponds to "From". These newsgroup posts are discussion forums and they start with a from: someone@gmail.com line. We see this when looking at `train_df.head()`_

For each of the 2000 tokens we extracted from our training data, we will look up its vector from `embedding_vector`.

In [ ]:
vocabulary = text_vectorization_glove.get_vocabulary()
word_index = dict(zip(vocabulary, range(len(vocabulary))))

counter = 0
embedding_matrix = np.zeros((max_tokens, embedding_dim))
for word, i in word_index.items():
    if i < max_tokens:
        embedding_vector = embeddings_index.get(word)
    if embedding_vector is not None:
        embedding_matrix[i] = embedding_vector
    else:
        counter += 1

embedding_matrix

array([[ 0.        ,  0.        ,  0.        , ...,  0.        ,
         0.        ,  0.        ],
       [ 0.        ,  0.        ,  0.        , ...,  0.        ,
         0.        ,  0.        ],
       [ 0.04656   ,  0.21318001, -0.0074364 , ...,  0.0090611 ,
        -0.20988999,  0.053913  ],
       ...,
       [-0.038683  ,  0.32398999,  0.10977   , ...,  0.10819   ,
        -0.22115   , -0.17591   ],
       [ 0.47946   ,  0.23098999, -0.11539   , ..., -0.26082999,
        -0.30032   ,  0.089238  ],
       [ 0.19551   , -0.24070001,  0.36401001, ..., -0.069489  ,
         0.32620001,  0.099097  ]])

The above matrix is of shape 2000 by 300. Each row corresponds to one of the 2000 tokens and the value in the row is the 300-dimensional vectorization of that token. We notice that the first two tokens above (index 0 and index 1) are mapped to the all zeros vector.

**Question:** What tokens are represented by index 0 and index 1?

<font color='red'>**Your Answer Here**</font>

_Replace this with your answer_

_They correspont to the [PAD] and [UNK] tokens_


## Part (b): Build the Model [10 Points]
Now, let's build the model. Fill in the missing code below.

* Create an embedding layer of the appropriate input and output dimensions. Initialize it to `keras.initializers.Constant(embedding_matrix)`, set `trainable=False` and `mask_zero=True`.
* Add a `GlobalAveragePooling1D` layer.
* Followed by a Dense layer with 128 neurons (ReLU activation), then a Dropout layer with probability of dropout 0.2. Follow this by another Dense layer of 128 neurons and a Dropout with 0.2 probability.


Your model should have 655,814 parameters out of which 55,814 are trainable.

In [ ]:
embedding_layer = keras.layers.Embedding(
    max_tokens, # input dimension
    embedding_dim, # output dimension
    embeddings_initializer= keras.initializers.Constant(embedding_matrix),
    trainable=False,
    mask_zero=True,
)

inputs = keras.Input(shape=(max_length,))

x = embedding_layer(inputs)
x = keras.layers.GlobalAveragePooling1D()(x)
x = keras.layers.Dense(128, activation="relu")(x)
x = keras.layers.Dropout(0.2)(x)
x = keras.layers.Dense(128, activation="relu")(x)
x = keras.layers.Dropout(0.2)(x)

outputs = keras.layers.Dense(y_train.shape[1], activation="sigmoid")(x) # output layer

glove_model = keras.Model(inputs, outputs)
glove_model.summary()

# Compile model using Adam
glove_model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy']
)

# Fit model on the training data with 10 epochs and batch size of 32
glove_model.fit(
    x=X_train_glove[:200], y=y_train[:200],
    epochs=20, batch_size=32,
    verbose=1,
)

print("\n*** Test accuracy with 200 Examples: %.3f ***\n" % glove_model.evaluate(x=X_test_glove, y=y_test)[1])

# Fit model on the training data with 10 epochs and batch size of 32
glove_model.fit(
    x=X_train_glove, y=y_train,
    epochs=20, batch_size=32,
    verbose=1,
)

print("\n*** Test accuracy with All Examples: %.3f ***\n" % glove_model.evaluate(x=X_test_glove, y=y_test)[1])

Model: "model_2"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_3 (InputLayer)        [(None, 400)]             0         
                                                                 
 embedding (Embedding)       (None, 400, 300)          600000    
                                                                 
 global_average_pooling1d (  (None, 300)               0         
 GlobalAveragePooling1D)                                         
                                                                 
 dense_4 (Dense)             (None, 128)               38528     
                                                                 
 dropout (Dropout)           (None, 128)               0         
                                                                 
 dense_5 (Dense)             (None, 128)               16512     
                                                           

Report the performance of our Glove Model above which uses the pre-trained word embeddings without fine-tuning. Comment on this performance. Please propose an explanation for why you think GloVe did not perform as well as BoW.

**<font color='red'>Your Answer Here</font>**

_Please replace the table below with your numbers_

* 200 Examples: 55.8%
* All Examples: 79.5%

_Please replace this text with your response_.

_GloVe without fine-tuning does not perform as well as BoW for both 200 examples and all examples. There are a couple possible explanations for this. GloVe vectors are trained on Wikipedia data, which may not accurate represent the meaning of words in the newsgroup dataset. Averaging the vectors for each word can drown out a lot of signal, if many words like "the" appear._

# Problem 3: GloVe with Fine Tuning [30 Points]

In problem 2, we did not fine-tune the word embeddings from GloVe. We simply downloaded them from the internet and used them as-is. In this question, we will fine-tune the embeddings to the training data set.

## Part (a): Why Fine Tuning is Needed [10 Points]

The goal of this part is to motivate why fine tuning of the word embeddings is necessary. In the process, we will understand why BoW is able to perform so well over GloVe.

The 20-newsgroup dataset consists of posts from a news discussion forum (from back in the day). Each of these posts look like an email, with a From: someone@email.com and subject lines. This is called the **header** of the post. Similarly, the email also has a **footer**, which typically consists of the author's name and affiliation. Consider one of the training examples shown below:

In [ ]:
print(train_df['text'][2101])

From: henry@zoo.toronto.edu (Henry Spencer)
Subject: Re: HLV for Fred (was Re: Prefab Space Station?)
Article-I.D.: zoo.C51875.67p
Organization: U of Toronto Zoology
Lines: 28

In article <C5133A.Gzx@news.cso.uiuc.edu> jbh55289@uxa.cso.uiuc.edu (Josh Hopkins) writes:
>>>Titan IV launches ain't cheap 
>>Granted. But that's because titan IV's are bought by the governemnt. Titan
>>III is actually the cheapest way to put a pound in space of all US expendable
>>launchers.
>
>In that case it's rather ironic that they are doing so poorly on the commercial
>market.  Is there a single Titan III on order?

The problem with Commercial Titan is that MM has made little or no attempt
to market it.  They're basically happy with their government business and
don't want to have to learn how to sell commercially.

A secondary problem is that it is a bit big.  They'd need to go after
multi-satellite launches, a la Ariane, and that complicates the marketing
task quite significantly.

They also had some pr

The header would be

```
From: henry@zoo.toronto.edu (Henry Spencer)
Subject: Re: HLV for Fred (was Re: Prefab Space Station?)
Article-I.D.: zoo.C51875.67p
Organization: U of Toronto Zoology
Lines: 28
```

and the footer is

```
All work is one man's work.             | Henry Spencer @ U of Toronto Zoology
                    - Kipling           |  henry@zoo.toronto.edu  utzoo!henry
```

Note that this document belongs to the topic **sci.space** (i.e. astronomy).

Recall the text vectorization layer from Problem 1(a). An interesting token that appears in our vocabulary space is the token `henryzootorontoedu`, appearing at index 1121. This token also appears in the example text above.

In [ ]:
idx = text_vectorization_glove.get_vocabulary().index('henryzootorontoedu')
print('Token number %d is henryzootorontoedu' % idx)

Token number 1121 is henryzootorontoedu


This token actually appears in quite a few of the training examples. Notice that all of the labels of these documents is 4, corresponding to the category **sci.space**

In [ ]:
train_df.loc[np.where(X_train_glove == idx)[0]]

,text,label
211,From: keithley@apple.com (Craig Keithley)\nSub...,4
219,From: henry@zoo.toronto.edu (Henry Spencer)\nS...,4
271,From: henry@zoo.toronto.edu (Henry Spencer)\nS...,4
271,From: henry@zoo.toronto.edu (Henry Spencer)\nS...,4
274,From: sysmgr@king.eng.umd.edu (Doug Mohney)\nS...,4
...,...,...
2874,From: dong@oakhill.sps.mot.com (Don M. Gibson)...,4
2960,From: henry@zoo.toronto.edu (Henry Spencer)\nS...,4
2960,From: henry@zoo.toronto.edu (Henry Spencer)\nS...,4
3197,From: Leigh Palmer <palmer@sfu.ca>\nSubject: R...,4


Looking at our GloVe embeddings, the embedding for `henryzootorontoedu` is the all zeros vector.

In [ ]:
print('The embedding for token %d is' % idx, embedding_matrix[idx, :])

The embedding for token 1121 is [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]


Using the information above regarding this token `henryzootorontoedu`, please explain why BoW has an advantage of GloVe (with no fine-tuning). What do you think will happen to the embedding of `henryzootorontoedu` if we do perform fine-tuning?

<font color='red'>**Your Answer**</font>

_Please replace this text with your answer_

_We see that the token `henryzootorontoedu` (Token 1121) is highly correlated with the sci.space category. The bag of words will learn this very quickly. However, GloVe with pre-trained embeddings will not learn this. Since `henryzootorontoedu` is not one of the 400,000 English words that we have an embedding for, it receives the vector of all zeros. Therefore, for GloVe, the presence of of `henryzootorontoedu` will have no impact even though the word is highly predictive of the sci.space class. When we fine-tune the embeddings, very likely the token `henryzootorontoedu` will have a non-zero embedding and have an embedding that is similar to other words from the sci.space class (e.g. Titan, satellite, Mars)_

## Part (b): Stripping the Header [10 Points]

It turns out that we can strip the header and footer from the training and testing examples pretty easily. For example, the same document above is now shown below:

In [ ]:
newsgroups_train_raw = fetch_20newsgroups(subset='train', categories = ['alt.atheism', 'talk.religion.misc', 'comp.graphics', 'sci.space', 'sci.med', 'rec.autos'], remove=('headers', 'footers', 'quotes'))
newsgroups_test_raw = fetch_20newsgroups(subset='test', categories = ['alt.atheism', 'talk.religion.misc', 'comp.graphics', 'sci.space', 'sci.med', 'rec.autos'], remove=('headers', 'footers', 'quotes'))

train_df_raw = pd.DataFrame({'text': newsgroups_train_raw.data, 'label': newsgroups_train_raw.target})
test_df_raw = pd.DataFrame({'text': newsgroups_test_raw.data, 'label': newsgroups_test_raw.target})

print(train_df_raw['text'][2101])


The problem with Commercial Titan is that MM has made little or no attempt
to market it.  They're basically happy with their government business and
don't want to have to learn how to sell commercially.

A secondary problem is that it is a bit big.  They'd need to go after
multi-satellite launches, a la Ariane, and that complicates the marketing
task quite significantly.

They also had some problems with launch facilities at just the wrong time
to get them started properly.  If memory serves, the pad used for the Mars
Observer launch had just come out of heavy refurbishment work that had
prevented launches from it for a year or so.

There have been a few CT launches.  Mars Observer was one of them.  So
was that stranded Intelsat, and at least one of its brothers that reached
orbit properly.


Now, copy the code from Problem 1(a), to train a BoW model, but this time adapt the corpus and use `train_df_raw` and `test_df_raw` instead of `train_df` and `test_df`.

In [ ]:
# First, we configure a Text Vectorization layer using the default
# standardization and multi-hot encoding

# Set the maximum number of tokens (which is the size of the vocabulary) to 1000
max_tokens = 2000

# Configure the text vectorization layer
text_vectorization = keras.layers.TextVectorization(
    max_tokens=max_tokens,
    output_mode = 'multi_hot',
    standardize = 'lower_and_strip_punctuation',
    ngrams=1,
)

# Let's adapt the Text Vectorization layer using the training corpus
text_vectorization.adapt(train_df_raw['text'])

# We vectorize our input with the adapted Text Vectorization layer
X_train_raw = text_vectorization(train_df_raw['text'])
X_test_raw = text_vectorization(test_df_raw['text'])

# Build a baseline NN model with one hidden layer that has 8 neurons.
inputs = keras.Input(shape=(max_tokens, )) # Size of the input layer is the size of the vocabulary
x = keras.layers.Dense(128, activation="relu")(inputs) # only hidden layer
outputs = keras.layers.Dense(y_train.shape[1], activation='softmax')(x)

bow_model_raw = keras.Model(inputs, outputs)
bow_model_raw.summary()

# Compile model using Adam
bow_model_raw.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy']
)

# Build a baseline NN model with one hidden layer that has 8 neurons.
inputs = keras.Input(shape=(max_tokens, )) # Size of the input layer is the size of the vocabulary
x = keras.layers.Dense(128, activation="relu")(inputs) # only hidden layer
outputs = keras.layers.Dense(y_train.shape[1], activation='softmax')(x)

bow_model_raw = keras.Model(inputs, outputs)
bow_model_raw.summary()

# Compile model using Adam
bow_model_raw.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy']
)

# Fit model on the training data with 10 epochs and batch size of 32
bow_model_raw.fit(
    x=X_train_raw[:200], y=y_train[:200],
    epochs=20, batch_size=32,
    verbose=1,
)

print("\n*** Test accuracy with 200 Examples: %.4f ***\n" % bow_model_raw.evaluate(x=X_test_raw, y=y_test)[1])

# Fit model on the training data with 10 epochs and batch size of 32
bow_model_raw.fit(
    x=X_train_raw, y=y_train,
    epochs=20, batch_size=32,
    verbose=1,
)

print("\n*** Test accuracy with All Examples % .4f ***\n" % bow_model_raw.evaluate(x=X_test_raw, y=y_test)[1])

Model: "model_3"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_4 (InputLayer)        [(None, 2000)]            0         
                                                                 
 dense_7 (Dense)             (None, 128)               256128    
                                                                 
 dense_8 (Dense)             (None, 6)                 774       
                                                                 
Total params: 256902 (1003.52 KB)
Trainable params: 256902 (1003.52 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________
Model: "model_4"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_5 (InputLayer)        [(None, 2000)]            0         
                                                                 


Fill in the table below with the test accuracy. Comment on how BoW's performance changes when trained and tested on the 20-newsgroup dataset _without headers and footers_. Does this support our hypothesis from Part (a)?

<font color='red'>**Your Answer Here**</font>

Accuracy Scores
* 200 Training Examples: 49.8%
* All Examples: 67.5%

_Please replace this with your answer_

_The performance has gone down significantly. This supports our hypothesis from (a). When we removed a lot of the emails and people's names from the text, BoW's performance declines significantly. This implies that BoW was previously using these names to perform classification._



## Part (c): Build a Model [10 Points]

Now, let's fine tune the GloVe embeddings for our particular dataset. Copy your code from Problem 2, but now change `trainable=True` in `embedding_layer`. Do not change anything else about the model.

In [ ]:
embedding_layer = keras.layers.Embedding(
    max_tokens,
    embedding_dim,
    embeddings_initializer= keras.initializers.Constant(embedding_matrix),
    trainable=True,
    mask_zero=True,
)

inputs = keras.Input(shape=(max_length,))
x = embedding_layer(inputs)

x = keras.layers.GlobalAveragePooling1D()(x)
x = keras.layers.Dense(128, activation="relu")(x) # only hidden layer
x = keras.layers.Dropout(0.2)(x)
x = keras.layers.Dense(128, activation="relu")(x) # only hidden layer
x = keras.layers.Dropout(0.2)(x)

outputs = keras.layers.Dense(y_train.shape[1], activation="sigmoid")(x) # output layer

glove_model_tuned = keras.Model(inputs, outputs)
glove_model_tuned.summary()

# Compile model using Adam
glove_model_tuned.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy']
)

# Fit model on the training data with 10 epochs and batch size of 32
glove_model_tuned.fit(
    x=X_train_glove[:200], y=y_train[:200],
    epochs=20, batch_size=32,
    verbose=1,
)

print("\n*** Test accuracy with 200 Examples %.4f ***\n" % glove_model_tuned.evaluate(x=X_test_glove, y=y_test)[1])

# Fit model on the training data with 10 epochs and batch size of 32
glove_model_tuned.fit(
    x=X_train_glove, y=y_train,
    epochs=20, batch_size=32,
    verbose=1,
)

print("\n*** Test accuracy with All Examples %.4f ***\n" % glove_model_tuned.evaluate(x=X_test_glove, y=y_test)[1])

Model: "model_5"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_6 (InputLayer)        [(None, 400)]             0         
                                                                 
 embedding_1 (Embedding)     (None, 400, 300)          600000    
                                                                 
 global_average_pooling1d_1  (None, 300)               0         
  (GlobalAveragePooling1D)                                       
                                                                 
 dense_11 (Dense)            (None, 128)               38528     
                                                                 
 dropout_2 (Dropout)         (None, 128)               0         
                                                                 
 dense_12 (Dense)            (None, 128)               16512     
                                                           

Report the performance of our Glove Model above with fine tuning. Comment on this performance and compare it to that of the GloVe model without fine tuning. Explain when fine-tuning can be helpful and/or hurtful.

**<font color='red'>Your Answer Here</font>**

_Please replace the table below with your numbers_

* 200 Examples: 68.2% (Your answer may vary from this)
* All Examples: 84.1%

_Please replace this text with your response_.

_Answer. Fine tuning made the performance better for All Examples and worse for 200 Examples. This makes sense because when we have only a few examples, fine-tuning just leads to a lot of overfitting. On the other hand, when we have many examples, fine-tuning can lead to a better embeddings that are tailored towards a particular classification task that cap and thus better overall performance._

The embedding for the token `henryzootorontoedu` is now no longer all 0's. Nice!

In [ ]:
print('The embedding for token %d is' % idx, glove_model_tuned.layers[1].weights[0][idx, :])

The embedding for token 1121 is tf.Tensor(
[ 0.04548174  0.07106574  0.12559451  0.01939604 -0.113691    0.07588904
 -0.1142942  -0.13795526 -0.00836113 -0.11620262  0.16121845 -0.05949525
  0.11007643 -0.12671092 -0.15571752  0.10972071  0.12101343  0.13156746
 -0.08381724  0.01628437 -0.08191272 -0.06775402 -0.08077831  0.15563102
  0.11042489  0.13917764 -0.10874247  0.10440496 -0.12844928  0.10008034
  0.15958744  0.09053869 -0.1862929   0.06595966 -0.06099972 -0.10755301
  0.11991861  0.10591989  0.11384564  0.10832665 -0.09684828 -0.06728903
  0.00923315  0.06172772 -0.1374209   0.07383049  0.03174763  0.12510315
 -0.05039083 -0.13083616 -0.04695299  0.05505094 -0.11015958 -0.10422369
 -0.13049425  0.07598031  0.16093646  0.0624091  -0.08583169 -0.07189132
 -0.09192856 -0.04763409 -0.0336896  -0.10931462  0.10193513 -0.06472947
  0.06214646 -0.13656795 -0.12682696 -0.07962221  0.1292712   0.05727947
  0.11996164  0.04719196  0.08600345  0.08034435  0.01130225  0.13435683
  0.0553

# Problem 4: BERT Transformer Model [15 Points]

Finally, we use will a famous transformer pre-trained model called [Bert](https://en.wikipedia.org/wiki/BERT_(language_model)). Just like how ResNet was a pre-trained model for computer vision (image processing), BERT is a pre-trained model for natural language processing.

In [ ]:
!pip install -q -U tensorflow-text ## install the package for NLP tasks in tensorflow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 22.2 MB/s eta 0:00:00


In [ ]:
import tensorflow_hub as hub
import tensorflow_text

bert_preprocess = 'https://tfhub.dev/tensorflow/bert_en_uncased_preprocess/3'

bert_layers = 12
bert_units = 768
bert_heads = 12

bert_encoder = f'https://tfhub.dev/tensorflow/bert_en_uncased_L-{bert_layers}_H-{bert_units}_A-{bert_heads}/4'

Let's look to some examples of the text processing required for BERT:

In [ ]:
bert_preprocess_model = hub.KerasLayer(bert_preprocess)

text_test = ['This is comedy as it once was and comparing this with the two remakes.']
text_preprocessed = bert_preprocess_model(text_test)
print(f'Word Ids   : {text_preprocessed["input_word_ids"]}')

text_test = ['Although I rated this movie a 2 for showing a complete lack of effort in trying to create a quality horror film it was a 10 on the unintentional funny scale.']
text_preprocessed = bert_preprocess_model(text_test)
print(f'Word Ids   : {text_preprocessed["input_word_ids"]}')

Word Ids   : [[  101  2023  2003  4038  2004  2009  2320  2001  1998 13599  2023  2007
   1996  2048 12661  2015  1012   102     0     0     0     0     0     0
      0     0     0     0     0     0     0     0     0     0     0     0
      0     0     0     0     0     0     0     0     0     0     0     0
      0     0     0     0     0     0     0     0     0     0     0     0
      0     0     0     0     0     0     0     0     0     0     0     0
      0     0     0     0     0     0     0     0     0     0     0     0
      0     0     0     0     0     0     0     0     0     0     0     0
      0     0     0     0     0     0     0     0     0     0     0     0
      0     0     0     0     0     0     0     0     0     0     0     0
      0     0     0     0     0     0     0     0]]
Word Ids   : [[  101  2348  1045  6758  2023  3185  1037  1016  2005  4760  1037  3143
   3768  1997  3947  1999  2667  2000  3443  1037  3737  5469  2143  2009
   2001  1037  2184  2006  1996  4

Notice how all examples start with the token `101` and end with `102` (and then followed by `[PAD]` tokens as indicated by 0). Those are special placeholder tokens that can be used for different purposes. For this exercise, we will only concern about the first one, and we will call this token the 'classification token', or `[CLS]`.

The two functions below are used to convert text into tokens (`bert_textvect`) and tokens into embeddings (`bert_features`). You do not need to understand exactly how the functions below work. Note that the following cell may take some time to run (around 5 minutes).

In [ ]:
max_length = 512
preprocessor = hub.load(bert_preprocess)
encoder = hub.KerasLayer(bert_encoder, trainable=False)

def bert_textvect(x):
    """
    Converts a list of strings into a list of tokens for BERT
    """
    input = keras.layers.Input(shape=(), dtype=tf.string)
    tokenized_input = hub.KerasLayer(preprocessor.tokenize)(input)
    bert_pack_inputs = hub.KerasLayer(preprocessor.bert_pack_inputs, arguments=dict(seq_length=max_length))
    output = bert_pack_inputs([tokenized_input])
    model = keras.Model(input, output)
    result = model.predict(x)
    return result

def bert_features(x):
    """
    Converts the list of tokens into 768-dimensional embeddings for BERT.
    """
    inputs = dict(
    input_word_ids=keras.layers.Input(shape=(max_length,), dtype=tf.int32),
    input_mask=keras.layers.Input(shape=(max_length,), dtype=tf.int32),
    input_type_ids=keras.layers.Input(shape=(max_length,), dtype=tf.int32),
    )

    output = encoder(inputs)['sequence_output'][:, 0, :]
    model = keras.Model(inputs, output)
    return model.predict(x)

X_bert_train = bert_textvect(train_df['text'])
X_bert_test = bert_textvect(test_df['text'])

features_train = bert_features(X_bert_train)
features_test = bert_features(X_bert_test)

features_train.shape

68/68 [==============================] - 91s 1s/step


(3222, 768)

Next, we'll use the BERT output embedding for the CLS token as input to train a simple neural net.

In [ ]:
input = keras.Input(shape=(bert_units, ))

x = keras.layers.Dense(128, activation='relu')(input)
x = keras.layers.Dropout(0.1)(x)
x = keras.layers.Dense(128, activation='relu')(x)
x = keras.layers.Dropout(0.1)(x)

output = keras.layers.Dense(y_train.shape[1], activation='softmax')(x)

# Model
bert_model = keras.Model(input, output)
bert_model.summary()

bert_model.compile(optimizer="adam",
              loss="categorical_crossentropy",
              metrics=["accuracy"])

bert_model.fit(
    x=features_train[:200], y=y_train[:200],
    epochs=20, batch_size=32,
    verbose=1,
)

print("\n*** Test accuracy with 200 Examples %.4f ***\n" % bert_model.evaluate(x=features_test, y=y_test)[1])

bert_model.fit(
    x=features_train, y=y_train,
    epochs=20, batch_size=32,
    verbose=1,
)

print("\n*** Test accuracy with All Examples %.4f ***\n" % bert_model.evaluate(x=features_test, y=y_test)[1])

Model: "model_10"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_15 (InputLayer)       [(None, 768)]             0         
                                                                 
 dense_14 (Dense)            (None, 128)               98432     
                                                                 
 dropout_4 (Dropout)         (None, 128)               0         
                                                                 
 dense_15 (Dense)            (None, 128)               16512     
                                                                 
 dropout_5 (Dropout)         (None, 128)               0         
                                                                 
 dense_16 (Dense)            (None, 6)                 774       
                                                                 
Total params: 115718 (452.02 KB)
Trainable params: 115718 

Fill in the table below with the accuracies from above. Comment on the performance of the model relative to the BoW and GloVe models. Does BERT suffer from the same shortcoming as GloVe regarding tokens like `henryzootorontoedu`?

<font color='red'>**Your Answer Here**</font>

BERT Model Accuracy
* 200 Examples: 72.9%
* All Examples: 80.5%

_Replace this with your answer_

_The performance of BERT with just 200 examples is an amazing 73.4%! Wow! None of the other models come close with only 200 examples. This truly shows the power of the transformer model. However, with all examples, BoW still performs the best, which is quite incredible for such a simple classification model. Intuitively, our news topic classification problem is just not that difficult of a problem. with enough training examples, BERT is a bit "overkill" and does not lead to any benefit over simple bag of words._


# Conclusion [5 Points]